Kmeans with all features except for highly correlated ones

In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, silhouette_score
from sklearn.model_selection import ParameterGrid
import matplotlib.pyplot as plt


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
root_dir = r'smaller_dataset.csv'
df = pd.read_csv(root_dir)

In [ ]:
columns_to_drop = ["Flow IAT Std",
                   "Bwd Segment Size Avg",
                   "Subflow Fwd Packets",
    'Flow Duration', 
    'Subflow Bwd Packets', 
    'Fwd Packet Length Max', 
    'Fwd Packet Length Min', 
    'Flow Packets/s', 
    'Flow IAT Min', 
    'Flow IAT Max', 
    'Bwd IAT Max', 
    'Bwd IAT Min', 
    'Fwd Header Length', 
    'ACK Flag Count', 
    'Packet Length Std',
    "Fwd IAT Max",
    "Idle Max",
    "Idle Min",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Max",
    'Average Packet Size', 
    'Fwd Segment Size Avg', 
    'Fwd IAT Max', 
    'Bwd Header Length', 
    'Packet Length Mean', 
    'CWR Flag Count', 
    'Average Packet Size',
    "Flow IAT Mean",
    "Active Max",
    "Bwd Bytes/Bulk Avg",
    'Fwd IAT Mean', 
    'Active Mean', 
    'Active Std',
    "Fwd Act Data Pkts"
]

df = df.drop(columns=columns_to_drop)
df = df.drop(columns="Label")
df.columns

In [ ]:
attacks_to_remove = [
    "spoofing_ARP Spoofing",
    "spoofing_DNS Spoofing",
    "sqlinjection",
    "XSS",
    "Uploading_Attack",
    "DoS_DoS SYN Flood",
    "DDoS_DDoS ACK Fragmentation"
]

df = df[~df["Attack_Type"].isin(attacks_to_remove)]

In [ ]:
categorical_columns = ["Src IP", 'Dst IP', "Src Port", "Dst Port", "Protocol"]  # replace with your actual column names

for col in categorical_columns:
    df[col] = df[col].astype('category')

In [ ]:
for col in ['Src IP', 'Dst IP', 'Src Port', 'Dst Port']:
    df[col + '_freq'] = df[col].map(df[col].value_counts())

df = pd.get_dummies(df, columns=['Protocol'], prefix='Proto')

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'],  format='%d/%m/%Y %I:%M:%S %p')

In [ ]:
def create_anomaly_label(row):
    if row['Attack_Type'] == 'Benign&Bruteforce_benign':  
        return 0
    else:
        return 1

df['Anomaly_Label'] = df.apply(create_anomaly_label, axis=1)

print(df[['Attack_Type', 'Anomaly_Label']].head())

In [ ]:
null_columns = df.isnull().sum()
null_columns = null_columns[null_columns > 0]  

inf_columns = df.columns[(df == np.inf).any() | (df == -np.inf).any()]
df = df.dropna()
df = df[~df.isin([np.inf, -np.inf]).any(axis=1)]
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [ ]:
labels = df['Anomaly_Label']
attack_types = df['Attack_Type']  #to track which attacks are detected

In [ ]:

numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(['Anomaly_Label'], errors='ignore')
X = df[numeric_cols].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3) Elbow curve to choose k
max_k = 10
inertias = []
ks = list(range(1, max_k+1))
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8,4))
plt.plot(ks, inertias, 'o-')
plt.xlabel("k")
plt.ylabel("Inertia")
plt.title("Elbow Method (k vs. inertia)")
plt.xticks(ks)
plt.grid(True)
plt.show()

# 4) Simple 2nd-derivative elbow detector
d1 = np.diff(inertias)
d2 = np.diff(d1)
elbow_k = int(np.argmin(d2) + 2)
print(f"Suggested k: {elbow_k}")

# 5) Fix k to elbow_k and define hyperparameter grid
k = elbow_k
param_grid = {
    "n_init":    [10, 20],
    "init":      ["k-means++", "random"],
    "algorithm": ["lloyd", "elkan"],
    "max_iter":  [300, 500],
}
hp_list = list(ParameterGrid(param_grid))

# 6) Run grid and write JSONL
output_file = "kmeans_hp_allFeat_results.jsonl"
with open(output_file, "w") as f_out:
    for hp in hp_list:
        km = KMeans(n_clusters=k, random_state=42, **hp)
        pred_clusters = km.fit_predict(X_scaled)

        # map each cluster → majority label
        cluster_map = {}
        for cid in range(k):
            idxs = np.where(pred_clusters == cid)[0]
            if len(idxs):
                majority = np.argmax(np.bincount(labels.to_numpy()[idxs]))
                cluster_map[cid] = int(majority)
            else:
                cluster_map[cid] = 0
        pred_labels = np.array([cluster_map[c] for c in pred_clusters])

        # confusion + metrics
        cm = confusion_matrix(labels, pred_labels, labels=[0,1])
        TN, FP, FN, TP = cm.ravel() if cm.size == 4 else (0,0,0,0)
        precision = TP/(TP+FP) if TP+FP else 0.0
        recall    = TP/(TP+FN) if TP+FN else 0.0
        f1        = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0

        # per-attack detection %
        attack_counts, attack_hits = {}, {}
        for true, pred, atk in zip(labels, pred_labels, attack_types):
            if true == 1:
                attack_counts[atk] = attack_counts.get(atk, 0) + 1
                if pred == 1:
                    attack_hits[atk] = attack_hits.get(atk, 0) + 1
        attack_pct = {
            atk: round(100 * attack_hits.get(atk, 0) / count, 2)
            for atk, count in attack_counts.items()
        }

        # write line
        out = {
            "n_clusters":        k,
            **hp,
            "inertia":          round(km.inertia_, 2),
            "precision":        round(precision, 4),
            "recall":           round(recall, 4),
            "f1_score":         round(f1, 4),
            "attack_detection_%": attack_pct
        }
        f_out.write(json.dumps(out) + "\n")

print(f"✅ Grid search results saved to {output_file}")


In [ ]:
import json

results = []
with open(output_file, "r") as f:
    for line in f:
        results.append(json.loads(line))

# Sort by F1 score (descending)
sorted_by_f1 = sorted(results, key=lambda x: x['f1_score'], reverse=True)

# Show top 5 results
for r in sorted_by_f1[:5]:
    # print each hyperparameter
    print("Hyperparameters:")
    print(f"  init     : {r['init']}")
    print(f"  n_init   : {r['n_init']}")
    print(f"  algorithm: {r['algorithm']}")
    print(f"  max_iter : {r['max_iter']}")
    print(f"F1 Score: {r['f1_score']}")
    print(f"Precision: {r['precision']}")
    print(f"Recall: {r['recall']}")
    print(f"Total Anomaly Detection Rate: {r['recall'] * 100:.2f}%")

    # Sort all attack types by detection % and print each
    all_attacks = sorted(r['attack_detection_%'].items(), key=lambda x: x[1], reverse=True)
    print("Attack detection rates:")
    for attack, pct in all_attacks:
        print(f"  {attack}: {pct}%")
